In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
from helper import plot_to_tensorboard, count_parameters, MLPClassifier
import pandas as pd
import hashlib
from pathlib import Path
import mlflow
import mlflow.pytorch
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
mlflow.set_experiment("Clasificador_Imagenes_HP_Search")

2026/05/21 17:04:49 INFO mlflow.tracking.fluent: Experiment with name 'Clasificador_Imagenes_HP_Search' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/ITBA/REDES%20NEURONALES/Tp1-Redes-Neuronales/mlruns/658278433750724451', creation_time=1779393889228, experiment_id='658278433750724451', last_update_time=1779393889228, lifecycle_stage='active', name='Clasificador_Imagenes_HP_Search', tags={}>

In [3]:
def log_classification_report(model, loader, writer, device, classes, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=classes)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    report_path = f"classification_report_{prefix}_epoch_{step}.txt"
    with open(report_path, "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(report_path)
    os.remove(report_path)

In [4]:
def evaluate(model, loader, writer, device, classes, criterion, epoch=None, prefix="val"):
    model.eval()  # 
    log_classification_report(model, loader, writer, device, classes, step=epoch, prefix=prefix)

    correct, total, loss_sum = 0, 0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [5]:
# Dataset que acepta lista de paths ya filtrados (no root_dir)
# NO aplana la imagen acá — el modelo ya tiene nn.Flatten()
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

        self.classes = sorted(list(set([Path(p).parent.name for p in self.image_paths])))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.labels = [self.class_to_idx[Path(p).parent.name] for p in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label  # <- NO aplana, el modelo tiene nn.Flatten()

In [6]:
train_dir = "data/Split_smol/train"
val_dir   = "data/Split_smol/val"

train_path_obj = Path(train_dir)
val_path_obj   = Path(val_dir)

all_train_paths = [p for p in train_path_obj.glob("**/*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]

def get_class(x):
    return x.parent.name

valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

files_val = []
for x in val_path_obj.rglob('*'):
    if x.is_file() and x.suffix.lower() in valid_extensions:
        try:
            with Image.open(x) as img:
                files_val.append((x, get_class(x), img.size, img.mode))
        except Exception:
            pass

df_val_completo = pd.DataFrame(files_val, columns=["path", "class", "resolution", "mode"])
df_val_completo = df_val_completo.sort_values(by="path").reset_index(drop=True)

# Split val → val (50%) + test (50%), por clase, reproducible
np.random.seed(42)
df_test = df_val_completo.groupby('class', group_keys=False).apply(
    lambda x: x.sample(frac=0.5, random_state=42)
)
df_val_recortado = df_val_completo.drop(df_test.index).reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

all_val_paths  = [Path(p) for p in df_val_recortado['path'].tolist()]
all_test_paths = [Path(p) for p in df_test['path'].tolist()]

# Hashes de train para filtrar leakage
train_hashes = {}
for p in all_train_paths:
    with open(p, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    train_hashes[file_hash] = p.name

# Filtrar TRAIN: foto negra + duplicado interno
foto_negra_train        = "ISIC_0031430.jpg"
duplicado_interno_train = "ISIC_0031039.jpg"
train_image_paths = [
    str(p) for p in all_train_paths
    if p.name != foto_negra_train and p.name != duplicado_interno_train
]

# Filtrar VAL contra TRAIN
val_image_paths = []
for p in all_val_paths:
    with open(p, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    if file_hash not in train_hashes:
        val_image_paths.append(str(p))

# Filtrar TEST contra TRAIN
test_image_paths = []
for p in all_test_paths:
    with open(p, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    if file_hash not in train_hashes:
        test_image_paths.append(str(p))

print(f"Train: {len(train_image_paths)} | Val: {len(val_image_paths)} | Test: {len(test_image_paths)}")

C:\Users\Sofia\AppData\Local\Temp\ipykernel_16752\3980641911.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test = df_val_completo.groupby('class', group_keys=False).apply(


Train: 695 | Val: 73 | Test: 75


In [7]:
# Crear directorio de logs de tensorboard
log_dir = "runs/experimento_skin_hp"
writer  = SummaryWriter(log_dir=log_dir)

In [ ]:
# PROBAR HP!!!!!!!!!!!

hparams_space = {
    "input_size":     [64],
    "batch_size":     [32],
    "lr":             [1e-4],
    "epochs":         50,
    "optimizer":      ["Adam"],
    "HFlip":          [0.5],
    "VFlip":          [0.5],           
    "rotate90":       [0.5],
    "RBContrast":     [0.2, 0.4],
    "CLAHE":          [0.0, 0.3],
    "coarse_dropout": [0.0, 0.3],
    "dropout":        [0.25],
    "es_patience":    5,
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelnbr = 0
for RBContrast in hparams_space["RBContrast"]:
    for CLAHE in hparams_space["CLAHE"]:
        for coarse_dropout in hparams_space["coarse_dropout"]:

            modelnbr += 1
            print(f"Modelo número: {modelnbr}", end="\r")

            hparams = {
                "model":          "MLPClassifier",
                "input_size":     hparams_space["input_size"][0],
                "batch_size":     hparams_space["batch_size"][0],
                "lr":             hparams_space["lr"][0],
                "epochs":         hparams_space["epochs"],
                "optimizer":      hparams_space["optimizer"][0],
                "HFlip":          hparams_space["HFlip"][0],
                "VFlip":          hparams_space["VFlip"][0],
                "rotate90":       hparams_space["rotate90"][0],
                "RBContrast":     RBContrast,
                "CLAHE":          CLAHE,
                "coarse_dropout": coarse_dropout,
                "loss_fn":        "CrossEntropyLoss",
                "train_dir":      train_dir,
                "val_dir":        val_dir,
                "es_patience":    hparams_space["es_patience"],
                "dropout":        hparams_space["dropout"][0],
            }

            input_size    = hparams["input_size"]
            batch_size    = hparams["batch_size"]
            lr            = hparams["lr"]
            optimizer_name = hparams["optimizer"]
            dropout       = hparams["dropout"]

            train_transform = A.Compose([
                A.Resize(input_size, input_size),
                A.HorizontalFlip(p=hparams["HFlip"]),
                A.VerticalFlip(p=hparams["VFlip"]),
                A.RandomRotate90(p=hparams["rotate90"]),
                A.RandomBrightnessContrast(p=RBContrast),
                A.CLAHE(p=CLAHE),
                A.CoarseDropout(max_holes=8, max_height=1, max_width=20, p=coarse_dropout),
                A.Normalize(),
                ToTensorV2()
            ])
            val_transform = A.Compose([
                A.Resize(input_size, input_size),
                A.Normalize(),
                ToTensorV2()
            ])

            train_dataset = CustomImageDataset(train_image_paths, transform=train_transform)
            val_dataset   = CustomImageDataset(val_image_paths,   transform=val_transform)

            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            val_loader   = DataLoader(val_dataset,   batch_size=batch_size)

            num_classes = len(train_dataset.classes)
            model = MLPClassifier(
                input_size=input_size ** 2 * 3,
                dropout=dropout,
                num_classes=num_classes
            ).to(device)

            criterion = nn.CrossEntropyLoss()
            optimizer = (
                optim.Adam(model.parameters(), lr=lr)
                if optimizer_name == "Adam"
                else optim.SGD(model.parameters(), lr=lr)
            )

            hparams["count_params"] = count_parameters(model)

            with mlflow.start_run():
                mlflow.log_params(hparams)

                best_val_acc    = 0
                best_val_loss   = 0
                best_train_acc  = 0
                best_train_loss = 0
                best_epoch      = 0

                for epoch in range(hparams["epochs"]):
                    model.train()
                    running_loss = 0.0
                    correct, total = 0, 0

                    for images, labels in train_loader:
                        images, labels = images.to(device), labels.to(device)

                        optimizer.zero_grad()
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                        loss.backward()
                        optimizer.step()

                        running_loss += loss.item()
                        _, preds = torch.max(outputs, 1)
                        correct += (preds == labels).sum().item()
                        total   += labels.size(0)

                    train_loss = running_loss / len(train_loader)
                    train_acc  = 100.0 * correct / total
                    val_loss, val_acc = evaluate(
                        model, val_loader, writer, device,
                        train_dataset.classes, criterion,
                        epoch=epoch, prefix="val"
                    )

                    writer.add_scalar("train/loss",     train_loss, epoch)
                    writer.add_scalar("train/accuracy", train_acc,  epoch)

                    mlflow.log_metrics({
                        "train_loss":     train_loss,
                        "train_accuracy": train_acc,
                        "val_loss":       val_loss,
                        "val_accuracy":   val_acc
                    }, step=epoch)

                    if val_acc > best_val_acc:
                        best_val_acc    = val_acc
                        best_val_loss   = val_loss
                        best_train_acc  = train_acc
                        best_train_loss = train_loss
                        best_epoch      = epoch
                        torch.save(model.state_dict(), "mlp_model.pth")
                        mlflow.log_artifact("mlp_model.pth")
                        mlflow.pytorch.log_model(model, artifact_path="pytorch_model")

                    elif epoch > best_epoch + hparams["es_patience"]:
                        break

                mlflow.log_metrics({
                    "best_train_loss": best_train_loss,
                    "best_train_acc":  best_train_acc,
                    "best_val_loss":   best_val_loss,
                    "best_val_acc":    best_val_acc,
                    "best_epoch":      best_epoch
                }, step=epoch + 1)

print(f"\nBúsqueda terminada. Se entrenaron {modelnbr} modelos.")

c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", resul

KeyboardInterrupt: 